In [0]:
from pyspark.sql import functions as F

# load curated source tables
drug_products = spark.table("drug_products")
product_ingredients = spark.table("product_ingredients")
ingredients = spark.table("ingredients")

reports = spark.table("reports")
report_events = spark.table("report_events")
report_product_links = spark.table("report_product_links")

In [0]:
# cleaned up neo4j node ready version of drug_products table
neo4j_drug_products = (
    drug_products
    .select(
        F.col("product_ndc").alias("product_id"),
        F.col("brand_name").alias("product_name"),
        F.col("labeler_name").alias("manufacturer_name"),
        F.col("generic_name"),
        F.col("dosage_form"),
        F.col("product_type")
    )
    .dropna(subset=["product_id"])
    .dropDuplicates(["product_id"])
)

neo4j_drug_products.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("neo4j_drug_products")
display(neo4j_drug_products.limit(10))

product_id,product_name,manufacturer_name,generic_name,dosage_form,product_type
72932-009,Medicated Wipes Hemorroidal Wipes With Witch Hazel,"Wuhan Zonsen Medical Products Co., Ltd.",WITCH HAZEL,CLOTH,HUMAN OTC DRUG
73656-017,White Glo Mouthwash Protection,WHITE GLO USA INC,SODIUM MONOFLUOROPHOSPHATE,KIT,HUMAN OTC DRUG
74375-050,70 Rubbing Ethyl Alcohol,Iron Lab SA de CV,Alcohol,LIQUID,HUMAN OTC DRUG
75712-943,"Avobenzone, Homosalate, Octisalate",Old East Main Co.,"Avobenzone, Homosalate, Octisalate",SPRAY,HUMAN OTC DRUG
76162-520,Topcare Dual Action Back Pain,Topco Associates LLC,"acetaminophen, ibuprofen","TABLET, FILM COATED",HUMAN OTC DRUG
76420-413,prednisone,"Asclemed USA, Inc.",PREDNISONE,TABLET,HUMAN PRESCRIPTION DRUG
76420-574,Progesterone,"Asclemed USA, Inc.",Progesterone,CAPSULE,HUMAN PRESCRIPTION DRUG
76891-142,HAND SANITIZER SANDALWOOD WAVES,SCENT THEORY PRODUCTS LLC,ALCOHOL,SPRAY,HUMAN OTC DRUG
79481-0599,Antifungal Miconazole Liquid Continuous,Meijer Distribution Inc,Miconazole Nitrate,"AEROSOL, SPRAY",HUMAN OTC DRUG
79481-1205,Meijer Fiber Caplets,"MEIJER DISTRIBUTION, INC",Calcium Polycarbophil,"TABLET, FILM COATED",HUMAN OTC DRUG


In [0]:
from pyspark.sql import functions as F

neo4j_manufacturers = (
    neo4j_drug_products
    .select(F.col("manufacturer_name"))
    .dropna(subset=["manufacturer_name"])
    .dropDuplicates(["manufacturer_name"])
    .withColumn(
        "manufacturer_id",
        F.concat(F.lit("MFR_"), F.md5(F.upper(F.trim(F.col("manufacturer_name"))))
        )
    )
    .select(
        "manufacturer_id", 
        "manufacturer_name"
    )
)

neo4j_manufacturers.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("neo4j_manufacturers")
display(neo4j_manufacturers.limit(10))

manufacturer_id,manufacturer_name
MFR_2886e2292a70328a0f173ec74c457e86,"Wuhan Zonsen Medical Products Co., Ltd."
MFR_b9555c15dd2f95766f21cd1eab25a554,WHITE GLO USA INC
MFR_4e69b1a6b32e96b11792f391f8c851ca,Iron Lab SA de CV
MFR_a33947d03bad9fc614a6b3cc58e20cfe,Old East Main Co.
MFR_816772c66cb62c5cb8b3171d61e8c80c,Topco Associates LLC
MFR_a4d78cf74066bce2759d27ccfb19d848,"Asclemed USA, Inc."
MFR_81ab075520254c701b7ee8c9c0531276,SCENT THEORY PRODUCTS LLC
MFR_b9e0ba123ee1323ed47a464c5fd70e08,Meijer Distribution Inc
MFR_2eccfc6e79f7812abbc2c7eb75656736,"MEIJER DISTRIBUTION, INC"
MFR_641f55245de768e80ab56a003b1a31d6,WALMART INC.


In [0]:
# cleaned up neo4j node ready version of ingredients table with dynamically generated ingredient ids

neo4j_ingredients = (
    ingredients
    .select(F.col("ingredient_name"))
    .dropna(subset=["ingredient_name"])
    .dropDuplicates(['ingredient_name'])
    .withColumn(
        "ingredient_id", 
        F.concat(F.lit("ING_"), F.md5(F.upper(F.trim(F.col("ingredient_name")))))
    )
)      

neo4j_ingredients.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("neo4j_ingredients")
display(neo4j_ingredients.limit(10))

ingredient_name,ingredient_id
PREDNISONE,ING_71a195762786d97199da64e4aa1b5ad7
ALLOPURINOL,ING_3314f326795fce4832401a57685de66d
DIEFFENBACHIA SEGUINE,ING_e22a1e2769eb515fb50660414e161d48
"MAGNESIUM PHOSPHATE, DIBASIC TRIHYDRATE",ING_48e04ba86cd5cd0eb94466af85506057
SALICYLIC ACID,ING_24a3fc3b184f6856d4e263fcc21acaa6
NOSCAPINE,ING_ed9c7f34e75c0ff7f953999afdd9617c
RALTEGRAVIR POTASSIUM,ING_d3fa2879efb8ab75eb37eea077de2b08
FENTANYL,ING_882387afebd5bdeeeec447d44868a238
PULSATILLA VULGARIS,ING_0222cc44681bd229640b1257583655d2
TORSEMIDE,ING_5c840392821c45cc87ac137c2ff44f66


In [0]:
# cleaned up neo4j node ready version of reports table
neo4j_reports = (
    reports
    .select(
        F.col("report_id"),
        F.col("report_date"),
        F.col("serious"),
        F.col("serious_label"),
        F.col("occurcountry")
    )
    .dropna(subset=["report_id"])
    .dropDuplicates(["report_id"])
)

neo4j_reports.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("neo4j_reports")
display(neo4j_reports.limit(10))

report_id,report_date,serious,serious_label,occurcountry
26059246,2025-11-18,1,SERIOUS,US
26170774,2025-12-17,1,SERIOUS,EU
26114683,2025-12-02,1,SERIOUS,US
26216514,2025-12-31,null,UNKNOWN,US
26068263,2025-11-20,2,NOT_SERIOUS,null
25948172,2025-10-21,1,SERIOUS,null
25953828,2025-10-23,2,NOT_SERIOUS,US
26098089,2025-11-27,1,SERIOUS,EU
24378355,2024-09-30,1,SERIOUS,CA
26174447,2025-12-18,null,UNKNOWN,US


In [0]:
# cleaned up neo4j node ready version of report_events table with dynamically generated event ids
neo4j_adverse_events = (
    report_events
    .select(F.col("event_name"))
    .dropna(subset=["event_name"])
    .dropDuplicates(['event_name'])
    .withColumn(
        "event_id", 
        F.concat(F.lit("EVT_"), F.md5(F.upper(F.trim(F.col("event_name")))))
    )
)    

neo4j_adverse_events.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("neo4j_adverse_events")
display(neo4j_adverse_events.limit(10))

event_name,event_id
NAUSEA,EVT_04fb35783bcf265052d41bfcd5d65a65
RASH,EVT_fb8ad20e6220daafea11bd5d03bc4693
PRODUCTIVE COUGH,EVT_8df0a52cdf1437bb00952ee81344d634
CONFUSIONAL STATE,EVT_3e76f15bd50f2e360a3310ece400558c
BLOOD LACTIC ACID INCREASED,EVT_1610d0b45feed7c80d45504add5e82fe
DRUG INEFFECTIVE,EVT_85d7c08f6db88b005f4f576c33205c85
ARTHRALGIA,EVT_16fda5179395e41c6adc0e9bfd12f5e9
HEADACHE,EVT_a648756546a381af72d01a17f853905f
HEPATIC LESION,EVT_581df0598ba7ecd6a21df1b93bac0d1b
INJECTION SITE MASS,EVT_9a60227f658dfbdccab5b1953be0ed3d


In [0]:
# establishing "HAS_INGREDIENT" relationship, joining product_ingredients and neo4j_ingredients to link product_ndc with ingredient_id
neo4j_rel_has_ingredient = (
    product_ingredients.alias("pi")
    .join(
        neo4j_ingredients.alias("i"),
        F.col("pi.ingredient_name") == F.col("i.ingredient_name"), "inner"
    )
    .select(
        F.col("pi.product_ndc").alias("from_product_id"),
        F.col("i.ingredient_id").alias("to_ingredient_id"),
        F.col("pi.ingredient_strength").alias("strength")
    )
    .dropna(subset=["from_product_id", "to_ingredient_id"])
    .dropDuplicates()
)

neo4j_rel_has_ingredient.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("neo4j_rel_has_ingredient")
display(neo4j_rel_has_ingredient.limit(10))

from_product_id,to_ingredient_id,strength
79903-134,ING_14294fbbc00f13df767ea45394c84066,10 mg/1
82757-204,ING_b1456f8cf62e0c8ceb555e63fa5e2165,30 mg/mL
85766-129,ING_d22e625bc38875e1c6d647d26b29c6ce,1 mg/100uL
0110-4839,ING_113bd5de78157a803960e9a7882dcbc2,14.5 mg/1
51552-0053,ING_24a3fc3b184f6856d4e263fcc21acaa6,1 g/g
63238-2500,ING_3e031f0eede9d06556c6038b74044f0c,1 kg/kg
66651-902,ING_191af386b77859604ce638f0a6525d1f,1 kg/kg
0093-6817,ING_886c9a1925aaca7440d57c32e1faefca,1 mg/2mL
30698-453,ING_a18241815241876d1c4521c4a36e5e46,20 mg/1
37662-2861,ING_18cedf73d4651d672a594fc6fef911f1,10 [hp_M]/1


In [0]:
# establishing "MANUFACTURED_BY" relationship joining neo4j_drug_products and neo4j_manufacturers to link product_ids with manufacturer_ids
neo4j_rel_manufactured_by = (
    neo4j_drug_products.alias("p")
    .join(
        neo4j_manufacturers.alias("m"),
        F.upper(F.trim(F.col("p.manufacturer_name"))) == F.upper(F.trim(F.col("m.manufacturer_name"))),
        "inner"
    )
    .select(
        F.col("p.product_id").alias("from_product_id"),
        F.col("m.manufacturer_id").alias("to_manufacturer_id")
    )
    .dropna(subset=["from_product_id", "to_manufacturer_id"])
    .dropDuplicates()
)

neo4j_rel_manufactured_by.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("neo4j_rel_manufactured_by")
display(neo4j_rel_manufactured_by.limit(10))

from_product_id,to_manufacturer_id
73656-017,MFR_b9555c15dd2f95766f21cd1eab25a554
82126-001,MFR_90b62bf855e1f5b7b463194092c11aab
0597-0575,MFR_133619dc88ec30f1282cf5421dced3fb
83819-102,MFR_2a968d16347e5bce5641ee56fc74f385
85766-123,MFR_bea1d0592a3c0fb493c35b195c800e66
62512-0018,MFR_920c0090b7ad790b9a2170326dd9bf85
13709-319,MFR_685919bf9cd398e0f7d8ce3ebc180278
31722-013,MFR_5e54a077ecbc3657b3e4b41cefced2bf
50090-5967,MFR_e68e6520e12872f8fafe1d5c642ccbf6
0220-4866,MFR_86b58100d86bb5c06483c4af86667542


In [0]:
# establishing "HAS_EVENT" relationship joining report_events and neo4j_adverse_events to link report_id with event_id
neo4j_rel_has_event = (
    report_events.alias("re")
    .join(
        neo4j_adverse_events.alias("e"),
        F.col("re.event_name") == F.col("e.event_name"), "inner"
    )
    .select(
        F.col("re.report_id").alias("from_report_id"),
        F.col("e.event_id").alias("to_event_id"),
        F.col("re.event_outcome"),
        F.col("re.event_outcome_label")
    )
    .dropna(subset=["from_report_id", "to_event_id"])
    .dropDuplicates()
)

neo4j_rel_has_event.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("neo4j_rel_has_event")
display(neo4j_rel_has_event.limit(10))

from_report_id,to_event_id,event_outcome,event_outcome_label
26159878,EVT_fbaacf623e96493e352e9099f39a0a59,3,NOT_RECOVERED
26109981,EVT_1ddc1f35e125ef110f1a1ff1294ca31d,6,UNKNOWN
26111481,EVT_281878cfc43e03663824cb2729256875,6,UNKNOWN
26021290,EVT_16fda5179395e41c6adc0e9bfd12f5e9,6,UNKNOWN
26210973,EVT_75adf030a0e7a6d0138f45320643e9a2,3,NOT_RECOVERED
25937402,EVT_85194126e7d643fd38b3952bf7e987fd,6,UNKNOWN
25527358,EVT_1169cdc8215ee00b59902195798049f8,3,NOT_RECOVERED
26138345,EVT_e29ee2c33ea4be0debe90c2b422b170e,2,RECOVERING
16008854,EVT_fee77740db1aefcbf8e1cf2018df3564,5,FATAL
16008854,EVT_f77d0e4c406bbd3bc652aa856e9962f1,5,FATAL


In [0]:
# cleaned up neo4j relationship ready version of report_product_links to establish "MENTIONS_PRODUCT" relationship between adverse event reports and ndc products
neo4j_rel_mentions_product = (
    report_product_links
    .filter(F.col("drug_characterization") == 1)
    .select(
        F.col("report_id").alias("from_report_id"),
        F.col("product_ndc").alias("to_product_id"),
        F.col("drug_name"),
        F.col("drug_characterization"),
        F.col("drug_characterization_label")
    )
    .dropna(subset=["from_report_id", "to_product_id"])
    .dropDuplicates()
)

neo4j_rel_mentions_product.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("neo4j_rel_mentions_product")
display(neo4j_rel_mentions_product.limit(10))

from_report_id,to_product_id,drug_name,drug_characterization,drug_characterization_label
25979480,60505-4534,LENALIDOMIDE,1,PRIMARY_SUSPECT_DRUG
25979480,60505-4536,LENALIDOMIDE,1,PRIMARY_SUSPECT_DRUG
25979480,70710-1032,LENALIDOMIDE,1,PRIMARY_SUSPECT_DRUG
25979480,70771-1680,LENALIDOMIDE,1,PRIMARY_SUSPECT_DRUG
25979480,59651-342,LENALIDOMIDE,1,PRIMARY_SUSPECT_DRUG
25979480,0574-1415,LENALIDOMIDE,1,PRIMARY_SUSPECT_DRUG
25979480,0378-1942,LENALIDOMIDE,1,PRIMARY_SUSPECT_DRUG
25979480,60219-1716,LENALIDOMIDE,1,PRIMARY_SUSPECT_DRUG
25979480,60219-1717,LENALIDOMIDE,1,PRIMARY_SUSPECT_DRUG
26041498,59148-035,BREXPIPRAZOLE,1,PRIMARY_SUSPECT_DRUG


VALIDATION CODE:

In [0]:
# check row counts
print("neo4j_drug_products:", neo4j_drug_products.count())
print("neo4j_manufacturers:", neo4j_manufacturers.count())
print("neo4j_ingredients:", neo4j_ingredients.count())
print("neo4j_reports:", neo4j_reports.count())
print("neo4j_adverse_events:", neo4j_adverse_events.count())

print("neo4j_rel_has_ingredient:", neo4j_rel_has_ingredient.count())
print("neo4j_rel_manufactured_by:", neo4j_rel_manufactured_by.count())
print("neo4j_rel_has_event:", neo4j_rel_has_event.count())
print("neo4j_rel_mentions_product:", neo4j_rel_mentions_product.count())

neo4j_drug_products: 131085
neo4j_manufacturers: 9536
neo4j_ingredients: 7641
neo4j_reports: 12000
neo4j_adverse_events: 4608
neo4j_rel_has_ingredient: 212599
neo4j_rel_manufactured_by: 131085
neo4j_rel_has_event: 54462
neo4j_rel_mentions_product: 1218649


In [0]:
%sql
--  ensure no duplicate node IDs

SELECT product_id, COUNT(*) AS cnt FROM neo4j_drug_products GROUP BY product_id HAVING cnt > 1;

SELECT manufacturer_id, COUNT(*) AS cnt FROM neo4j_manufacturers GROUP BY manufacturer_id HAVING cnt > 1;

SELECT ingredient_id, COUNT(*) AS cnt FROM neo4j_ingredients GROUP BY ingredient_id HAVING cnt > 1;

SELECT report_id, COUNT(*) AS cnt FROM neo4j_reports GROUP BY report_id HAVING cnt > 1;

SELECT event_id, COUNT(*) AS cnt FROM neo4j_adverse_events GROUP BY event_id HAVING cnt > 1;

event_id,cnt


In [0]:
%sql
--  ensure no null relationship endpoints

SELECT * FROM neo4j_rel_has_ingredient WHERE from_product_id IS NULL OR to_ingredient_id IS NULL;

SELECT * FROM neo4j_rel_manufactured_by WHERE from_product_id IS NULL OR to_manufacturer_id IS NULL;

SELECT * FROM neo4j_rel_has_event WHERE from_report_id IS NULL OR to_event_id IS NULL;

SELECT * FROM neo4j_rel_mentions_product WHERE from_report_id IS NULL OR to_product_id IS NULL;

from_report_id,to_product_id,drug_name,drug_characterization,drug_characterization_label


In [0]:
# load neo4j-ready node/relationship tables into csv files
neo4j_drug_products.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/abbvie_demo_files/neo4j_export/neo4j_drug_products")

neo4j_manufacturers.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/abbvie_demo_files/neo4j_export/neo4j_manufacturers")

neo4j_ingredients.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/abbvie_demo_files/neo4j_export/neo4j_ingredients")

neo4j_reports.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/abbvie_demo_files/neo4j_export/neo4j_reports")

neo4j_adverse_events.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/abbvie_demo_files/neo4j_export/neo4j_adverse_events")

neo4j_rel_has_ingredient.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/abbvie_demo_files/neo4j_export/neo4j_rel_has_ingredient")

neo4j_rel_manufactured_by.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/abbvie_demo_files/neo4j_export/neo4j_rel_manufactured_by")

neo4j_rel_has_event.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/abbvie_demo_files/neo4j_export/neo4j_rel_has_event")

neo4j_rel_mentions_product.coalesce(1).write.mode("overwrite").option("header", True).csv("/Volumes/workspace/default/abbvie_demo_files/neo4j_export/neo4j_rel_mentions_product")